# Threads, processes, and the Python GIL

**Core notebook, about 25 minutes.** Predict each result before running it. The goal is to choose an execution model, not to memorize one scheduler as always faster.

## Mental model

```text
one process                         several processes
+-----------------------------+    +-----------+  +-----------+
| one interpreter and one GIL |    | interpreter|  | interpreter|
| thread  thread  thread      |    | and GIL    |  | and GIL    |
| shared memory               |    | own memory |  | own memory |
+-----------------------------+    +-----------+  +-----------+
```

- Threads share memory and have low communication overhead.
- In standard CPython, the GIL allows only one thread to execute Python bytecode at a time.
- Processes can run Python bytecode on separate cores, but startup and data serialization cost time and memory.
- NumPy and compiled Numba functions may release the GIL during heavy work.

In [ ]:
import os
import time
import dask
from dask import delayed
from workloads import fib, sleepy

# A teaching benchmark needs enough work to show the model, not 128 tiny tasks.
n_workers = min(4, os.cpu_count() or 1)
print("Available logical CPUs:", os.cpu_count())
print("Workers used in this notebook:", n_workers)

## 1. CPU-bound pure Python

**Predict first:** which will be faster for several recursive Fibonacci calculations, threads or processes? What cost could make the result less clear for a very small task?

In [ ]:
cpu_tasks = [delayed(fib)(32) for _ in range(n_workers)]

started = time.perf_counter()
thread_results = dask.compute(
    *cpu_tasks, scheduler="threads", num_workers=n_workers
)
thread_time = time.perf_counter() - started

started = time.perf_counter()
process_results = dask.compute(
    *cpu_tasks, scheduler="processes", num_workers=n_workers
)
process_time = time.perf_counter() - started

assert thread_results == process_results
print(f"Threads:   {thread_time:.3f} s")
print(f"Processes: {process_time:.3f} s")

<details><summary>Interpretation</summary>

Pure-Python CPU work is constrained by the GIL when run in threads. Separate processes have separate interpreters and GILs, so they can run on several cores. For tiny tasks, process startup and serialization can outweigh that advantage.

</details>

## 2. Waiting work

`time.sleep` represents waiting for a file or network response and releases the GIL. **Predict first:** which scheduler should have less overhead here?

In [ ]:
waiting_tasks = [delayed(sleepy)(0.5) for _ in range(n_workers)]

started = time.perf_counter()
thread_wait_results = dask.compute(
    *waiting_tasks, scheduler="threads", num_workers=n_workers
)
thread_wait_time = time.perf_counter() - started

started = time.perf_counter()
process_wait_results = dask.compute(
    *waiting_tasks, scheduler="processes", num_workers=n_workers
)
process_wait_time = time.perf_counter() - started

assert thread_wait_results == process_wait_results
print(f"Threads:   {thread_wait_time:.3f} s")
print(f"Processes: {process_wait_time:.3f} s")

## Your turn: explain the evidence

**12 minutes.** Discuss with a neighbor:

1. Did both results match your predictions?
2. What would happen if every task received a 2 GB array?
3. Which scheduler would you try for a NumPy-heavy task that releases the GIL?
4. What measurement would you collect before changing a real program?

Blue sticky note means you need help. Yellow means you can explain your choice.

## Takeaway

- Use threads for waiting work or compiled work that releases the GIL, especially when sharing large in-memory data matters.
- Try processes for coarse, CPU-bound pure-Python tasks when serialization is affordable.
- Control worker counts. More workers can increase contention and memory use.
- Benchmark the real workload because task size, libraries, and data movement affect the result.

Next: [`../5_dask/1_delayed.ipynb`](../5_dask/1_delayed.ipynb).